[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arrjon/BayesFlowTutorial/blob/main/02_SIR_Diagnostics_Solution.ipynb)

# BayesFlow 2 — SIR Epidemic Models & Diagnostics — Solution

**Difficulty:** 🟡 Intermediate

> ✅ This is the **completed** notebook. Every `TODO` from the exercise is filled in. There is usually more than one good answer — yours may differ and still be correct.

In [ ]:
# ▶️  RUN THIS FIRST.  Installs this tutorial project and all its dependencies
#     (BayesFlow, JAX, Pyro, ...) from GitHub, as declared in pyproject.toml (~1-2 min).
import sys, subprocess

print("Installing the tutorial and its dependencies — this takes ~1-2 min ...")
_res = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/arrjon/BayesFlowTutorial.git"],
    capture_output=True, text=True,
)
if _res.returncode != 0:
    print(_res.stdout); print(_res.stderr)
    raise SystemExit("Install failed — see the pip output above.")

# NOTE: pip may internally warn that google-colab pins pandas==2.2.2 while BayesFlow
# needs pandas>=2.3.3. That warning is harmless and unavoidable (the tutorial doesn't use
# google-colab's pandas features).
print("✅ Setup complete.")
print("   If the NEXT cell raises a numpy/pandas import error, click "
      "Runtime ▸ Restart session and re-run.")

In [ ]:
import os
# BayesFlow runs on JAX, PyTorch or TensorFlow. We pick JAX here.
os.environ["KERAS_BACKEND"] = "jax"
import bayesflow as bf

> 📚 **Where to read more**
> - BayesFlow website & docs: <https://bayesflow.org>
> - Example gallery: <https://bayesflow.org/main/examples.html>
> - BayesFlow guide: https://bayesflow.org/v2.0.12/user_guide/diagnostics.html

## The SIR model

We infer parameters of a stochastic epidemiological model. Individuals are **S**usceptible, **I**nfected or **R**ecovered, with transmission rate $\lambda$ and recovery rate $\mu$:

$$\frac{dS}{dt} = -\lambda \frac{S I}{N},\quad \frac{dI}{dt} = \lambda \frac{S I}{N} - \mu I,\quad \frac{dR}{dt} = \mu I.$$

We observe **daily reported new cases** for $T=14$ days. Reporting is imperfect: cases appear with a delay $D$ (an integer number of days) and are noisy, modelled with a **Negative-Binomial** observation model with dispersion $\psi$. The initial number of infected $I_0$ is also unknown. So we infer five parameters: $(\lambda, \mu, D, I_0, \psi)$.

Background reading:
- OutbreakFlow (Radev et al., 2021): <https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1009472>

This builds directly on the linear-regression starter — the pipeline (simulator → adapter → networks → workflow → diagnostics) is the same, only the model is richer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

RNG = np.random.default_rng(2026)

## 1. Prior & simulator

We will place the following prior distributions over the five model parameters, summarized in the table below:

$$
\begin{aligned}
& \text {Table 1. Description of model parameters and corresponding prior distributions}\\
&\begin{array}{lcl}
\hline \hline \text { Description} & \text { Symbol } & \text { Prior Distribution } \\
\hline \hline \text{Initial transmission rate} & \text{$\lambda$} & \text{$\textrm{LogNormal}(\log(0.4), 0.5)$} \\
\text{Recovery rate of infected individuals} & \text{$\mu$} & \text{$\textrm{LogNormal}(\log(1/8), 0.2)$} \\
\text{Reporting delay (lag)} & \text{$D$} & \text{$\textrm{LogNormal}(\log(8), 0.2)$} \\
\text{Number of initially infected individuals} & \text{$I_0$} & \text{$\textrm{Gamma}(2, 20)$} \\
\text{Dispersion of the negative binomial distribution} & \text{$\psi$} & \text{$\textrm{Exponential}(5)$} \\
\hline
\end{array}
\end{aligned}
$$

How did we come up with these priors? In this case, we rely on the domain expertise and previous research  (https://www.science.org/doi/10.1126/science.abb9789). In addition, the new parameter $\psi$ follows an exponential distribution, which restricts it to positive numbers. Below is the implementation of these priors:

In [ ]:
def prior():
    """A random draw from the joint prior over the five parameters."""
    lambd = RNG.lognormal(mean=np.log(0.4), sigma=0.5)
    mu    = RNG.lognormal(mean=np.log(1 / 8), sigma=0.2)
    D     = RNG.lognormal(mean=np.log(8), sigma=0.2)
    I0    = RNG.gamma(shape=2, scale=20)
    psi   = RNG.exponential(5)
    return {"lambd": lambd, "mu": mu, "D": D, "I0": I0, "psi": psi}

### The observation model

The simulator integrates the ODE to calculate the **expected** reported new cases per day and then adds Negative-Binomial reporting noise on top.

The helper `convert_params` converts the (mean, dispersion) parameterisation of the Negative-Binomial into the `(n, p)` form NumPy expects.

In [ ]:
def convert_params(mu, phi):
    """Helper function to convert mean/dispersion parameterization of a negative binomial to N and p,
    as expected by numpy's negative_binomial.

    See https://en.wikipedia.org/wiki/Negative_binomial_distribution#Alternative_formulations
    """

    r = phi
    var = mu + 1 / r * mu**2
    p = (var - mu) / var
    return r, 1 - p


def stationary_SIR(lambd, mu, D, I0, psi, N=83e6, T=14, eps=1e-5, rng=None):
    """Performs a forward simulation from the stationary SIR model given a random draw from the prior."""
    if rng is None:
        rng = RNG
    # Extract parameters and round I0 and D
    I0 = np.ceil(I0)
    D = int(round(D))

    # Initial conditions
    S, I, R = [N - I0], [I0], [0]

    # Reported new cases
    C = [I0]

    # Simulate T-1 timesteps
    for t in range(1, T + D):
        # Calculate new cases
        I_new = lambd * (I[-1] * S[-1] / N)

        # SIR equations
        S_t = S[-1] - I_new
        I_t = np.clip(I[-1] + I_new - mu * I[-1], 0.0, N)
        R_t = np.clip(R[-1] + mu * I[-1], 0.0, N)

        # Track
        S.append(S_t)
        I.append(I_t)
        R.append(R_t)
        C.append(I_new)

    C = np.clip(np.array(C[D:]), 0, N)
    reparam = convert_params(C + eps, psi)
    C_obs = rng.negative_binomial(reparam[0], reparam[1])
    return dict(cases=C_obs, raw_cases=C)

Combine prior and simulator, and sanity-check the output shapes.

In [ ]:
simulator = bf.make_simulator([prior, stationary_SIR])

test_sims = simulator.sample(batch_size=3)
print("lambd:", test_sims["lambd"].shape, " cases:", test_sims["cases"].shape)  # (3,1) (3,14)

### Prior predictive check

A principled Bayesian workflow always inspects the prior. Let's look at the joint prior of $\lambda$, $\mu$, $D$.

In [ ]:
prior_samples = simulator.simulators[0].sample(1000)
grid = bf.diagnostics.plots.pairs_samples(prior_samples, variable_keys=["lambd", "mu", "D"])

## 2. Adapter

The raw data are large integer counts and the parameters live on very different scales — not network-friendly. We:
- mark `cases` as a **time series** (`as_time_series`), giving it shape `(batch, T, 1)`,
- concatenate the five parameters into `inference_variables`,
- rename `cases` to `summary_variables` (they go through the summary network),
- `log1p`-transform the data.
- constrain the parameters to be positive (the network will learn to output unconstrained values, which are then transformed to the constrained space).

In [ ]:
adapter = (
    bf.adapters.Adapter()
    .convert_dtype("float64", "float32")
    .as_time_series("cases")
    .concatenate(["lambd", "mu", "D", "I0", "psi"], into="inference_variables")
    .rename("cases", "summary_variables")
    .log("summary_variables", p1=True)
    .constrain("inference_variables", lower=0)
)

adapted = adapter(simulator.sample(2))
print(adapted["summary_variables"].shape, adapted["inference_variables"].shape)  # (2,14,1) (2,5)

## 3. Networks & workflow

Because our data is a **sequence**, we use a small custom **GRU** summary network. Any Keras model can plug into BayesFlow by subclassing `SummaryNetwork`. The `@serializable` decorator lets us save/load the full approximator later. BayesFlow has a lot of summary network architectures built in, but here we illustrate how to build a custom one.

For the inference network we again use a `CouplingFlow`.

In [ ]:
@bf.utils.serialization.serializable("custom")
class GRU(bf.networks.SummaryNetwork):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gru = keras.layers.GRU(64)
        self.summary_projection = keras.layers.Dense(8)

    def call(self, time_series, **kwargs):
        """Compress (batch, T, 1) time series into (batch, 8) summaries."""
        summary = self.gru(time_series, **kwargs)
        summary = self.summary_projection(summary)
        return summary


summary_net = GRU()
inference_net = bf.networks.CouplingFlow(depth=2, transform="spline")

In [ ]:
workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_net,
    summary_network=summary_net,
)

## 4. Training (offline)

The simulator is fast, but to illustrate a realistic *low simulation budget* we pre-generate a fixed dataset and train **offline**. ~1 minute on Colab CPU.

In [ ]:
training_data = workflow.simulate(6000)
validation_data = workflow.simulate(300)

history = workflow.fit_offline(
    data=training_data, epochs=100, batch_size=64, validation_data=validation_data
)
f = bf.diagnostics.plots.loss(history)

## 5. Diagnostics

A converged loss is **not** proof of correct inference. We validate *in silico* (on fresh simulations where we know the truth) before ever touching real data.

First, draw posterior samples for 300 fresh test datasets.

In [ ]:
num_datasets, num_samples = 300, 1000
test_sims = workflow.simulate(num_datasets)
samples = workflow.sample(conditions=test_sims, num_samples=num_samples, batch_size=50)

### 5a. Simulation-based calibration (SBC)

If the posterior is well calibrated, the rank of each true value among the posterior draws is **uniform**. Rank **histograms** should look flat; the **ECDF-difference** should stay inside the band. (ECDF is usually easier to read for many parameters.)

> SBC papers: rank histograms <https://arxiv.org/abs/1804.06788> · ECDF <https://arxiv.org/abs/2103.10522>
>
> BayesFlow guide: https://bayesflow.org/v2.0.12/user_guide/diagnostics.html

In [ ]:
f = bf.diagnostics.plots.calibration_histogram(samples, test_sims)
f = bf.diagnostics.plots.calibration_ecdf(samples, test_sims, difference=True)

### 5b. Recovery & the non-identifiability lesson

Now compare posterior means to ground truth.

In [ ]:
f = bf.diagnostics.plots.recovery(samples, test_sims)

> 🔎 **Interpret before you scroll.** Look at the recovery plots for $\mu$ and $D$. They probably look *flat* — the estimate barely moves with the truth. **Is training broken?**
>
> No. From only 14 days of case counts, $\mu$ (recovery rate) and $D$ (reporting delay) are **weakly identified**: many parameter combinations produce near-identical data. No inference method — BayesFlow, MCMC, anything — can recover what the data doesn't contain. A flat recovery plot combined with **good calibration** is the signature of non-identifiability, not a bug. This is exactly why we run diagnostics.

### 5c. Automatic numerical diagnostics

The workflow can compute the 'big three' error metrics — NRMSE, calibration error, posterior contraction — in one call. **Posterior contraction near 0** for a parameter is another fingerprint of non-identifiability (the posterior is as wide as the prior).

In [ ]:
metrics = workflow.compute_default_diagnostics(test_data=test_sims, samples=samples)
metrics

In [ ]:
# The full graphical diagnostic panel in one call
figures = workflow.plot_default_diagnostics(
    test_data=test_sims, samples=samples,
    loss_kwargs={"figsize": (15, 3)}, recovery_kwargs={"figsize": (15, 3)},
    calibration_ecdf_kwargs={"figsize": (15, 3)}, z_score_contraction_kwargs={"figsize": (15, 3)},
)

## 6. Confronting real data: a posterior predictive check 📈

In-silico diagnostics tell us the network is well calibrated *for data that looks like our simulator*. The final test is **real data**.

We download the actual reported COVID-19 cases in Germany for the first two weeks of March 2020, infer the posterior with our trained network, and run a **posterior predictive check (PPC)**: draw parameters from the posterior, re-simulate case trajectories, and overlay them on the observed series. If the model is adequate, the observations should fall within the predictive band.

In [ ]:
import datetime, pandas as pd

def load_data():
    """Download daily reported COVID-19 cases in Germany, 1-14 March 2020 (needs internet)."""
    url = ("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/"
           "csse_covid_19_data/csse_covid_19_time_series/"
           "time_series_covid19_confirmed_global.csv")
    df = pd.read_csv(url)
    fmt = lambda d: f"{d.month}/{d.day}/{str(d.year)[2:4]}"
    b, e = datetime.date(2020, 3, 1), datetime.date(2020, 3, 15)
    cum = np.array(df.loc[df["Country/Region"] == "Germany", fmt(b):fmt(e)])[0]
    return np.diff(cum)  # daily new cases = differences of the cumulative counts

real_cases = load_data()

plt.figure(figsize=(9, 3))
plt.plot(real_cases, "-o", color="black")
plt.xlabel("day (from 1 March 2020)"); plt.ylabel("reported new cases")
plt.title("Real German COVID-19 cases, early March 2020");

### Posterior predictive check

The helper below draws re-simulations from the posterior (one trajectory per posterior sample) and overlays their median and credible bands on the observed data.

In [ ]:
def plot_ppc(param_df, obs_cases, sim_fn, logscale=True, color="#132a70", title="", seed=2026):
    """Overlay posterior re-simulations (from sim_fn) on the observed data."""
    rng = np.random.default_rng(seed)  # seed the reporting noise so the check is reproducible
    T = len(obs_cases)
    sims = []
    for row in param_df.values:
        try:
            sims.append(sim_fn(*row, rng=rng)["cases"])
        except ValueError:
            continue  # drop the rare out-of-support posterior draw (e.g. psi == 0)
    sims = np.array(sims)
    q50 = np.quantile(sims, [0.25, 0.75], axis=0)
    q90 = np.quantile(sims, [0.05, 0.95], axis=0)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(np.median(sims, axis=0), color=color, label="median re-simulation")
    ax.fill_between(range(T), q50[0], q50[1], color=color, alpha=0.4, label="50% CI")
    ax.fill_between(range(T), q90[0], q90[1], color=color, alpha=0.2, label="90% CI")
    ax.plot(obs_cases, "o--", color="black", alpha=0.8, label="observed")
    if logscale: ax.set_yscale("log")
    ax.set_xlabel("day"); ax.set_ylabel("cases"); ax.set_title(title); ax.legend()
    return fig

In [ ]:
post = workflow.sample(conditions={"cases": real_cases[None, :]}, num_samples=1000)
post_df = workflow.samples_to_data_frame(post)

f = plot_ppc(post_df, real_cases, sim_fn=stationary_SIR,
             title="Posterior predictive check — real German COVID-19 data")

> 🔎 **What to look for.** If the observed points mostly fall inside the predictive band, the SIR model is an adequate description of the early outbreak. Systematic structure in the residuals — for instance the observed series repeatedly leaving the band on the same weekday — would flag **model misspecification**: a real effect (such as reduced weekend reporting) that the simulator does not capture. The posterior can still look confident in that case, which is exactly why the PPC is the essential last check before trusting the results.

## 🎉 Summary

You trained an amortized estimator for a mechanistic time-series model, learned to **read** SBC / recovery / contraction diagnostics, recognised **non-identifiability**, and finally confronted the network with **real COVID-19 data** through a **posterior predictive check**.